# OSL Uni-Sign Training Pipeline

This notebook runs your training pipeline from one place.

Run order:
1. Install dependencies (one time)
2. Train OSL-Words (ISLR)
3. Prepare OSL-Sentences dataset (labels and splits)
4. Train OSL-Sentences (SLT)

In [33]:
import os
from datetime import datetime

# New clean run name
RUN_ID = datetime.now().strftime("osl_words_1gpu_%Y%m%d_%H%M%S")

# New experiment folder
EXP_DIR = f"/home/sign_lang_fyp_sp26/fyp/experiments/{RUN_ID}"
os.makedirs(EXP_DIR, exist_ok=True)

print("Fresh experiment folder:", EXP_DIR)

Fresh experiment folder: /home/sign_lang_fyp_sp26/fyp/experiments/osl_words_1gpu_20260417_013319


In [29]:
import subprocess
check_cmd = """
source ~/torchenv/bin/activate
cd ~/fyp/Uni-sign-main/Uni-Sign-main
python -c "
import torch
ckp = torch.load('checkpoints/csl_stage2_weight.pth', map_location='cpu')
params = sum(v.numel() for v in ckp['model'].values())
print(f'Checkpoint params: {params/1e6:.1f}M')
print(f'Number of keys: {len(ckp[\"model\"])}')
"
"""
result = subprocess.run(["bash", "-lc", check_cmd], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

Checkpoint params: 978.4M

Traceback (most recent call last):
  File "<string>", line 6, in <module>
NameError: name 'model' is not defined



In [35]:
import subprocess
import os
from pathlib import Path
from datetime import datetime

# -----------------------------
# Main project path
# -----------------------------
UNISIGN_DIR = Path.home() / "fyp" / "Uni-sign-main" / "Uni-Sign-main"

# -----------------------------
# Timestamped experiment folder
# -----------------------------
RUN_ID = datetime.now().strftime("osl_words_1gpu_%Y%m%d_%H%M%S")
EXP_DIR = Path.home() / "fyp" / "experiments" / RUN_ID
EXP_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Checkpoint output folder
# -----------------------------
CKPT_DIR = UNISIGN_DIR / "checkpoints" / RUN_ID
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = EXP_DIR / "train.log"
PID_FILE = EXP_DIR / "train.pid"

cmd = f"""
source ~/torchenv/bin/activate
cd {UNISIGN_DIR}

nohup python fine_tuning.py \
    --dataset OSL-Words \
    --task ISLR \
    --finetune checkpoints/csl_stage2_weight.pth \
    --output_dir {CKPT_DIR} \
    --epochs 30 \
    --lr 1e-4 \
    --warmup-epochs 3 \
    --gradient-accumulation-steps 1 \
    --batch-size 4 \
    --max_length 256 \
    --label_smoothing 0.1 \
    --num_workers 4 \
    --zero_stage 0 \
    > {LOG_FILE} 2>&1 &

echo $! > {PID_FILE}
echo "Training PID: $!"
"""

result = subprocess.run(["bash", "-lc", cmd], text=True, capture_output=True)

print(result.stdout)
print(result.stderr)

print("Experiment folder:", EXP_DIR)
print("Checkpoint folder:", CKPT_DIR)
print("Log file:", LOG_FILE)

Training PID: 3068169


Experiment folder: /home/sign_lang_fyp_sp26/fyp/experiments/osl_words_1gpu_20260417_013329
Checkpoint folder: /home/sign_lang_fyp_sp26/fyp/Uni-sign-main/Uni-Sign-main/checkpoints/osl_words_1gpu_20260417_013329
Log file: /home/sign_lang_fyp_sp26/fyp/experiments/osl_words_1gpu_20260417_013329/train.log


In [4]:
import re
import json
import argparse
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)


def parse_args():
    class Args:
        RUN_DIR = "/home/sign_lang_fyp_sp26/fyp/experiments/osl_words_1gpu_20260417_013329"
        PRED_CSV = "/home/sign_lang_fyp_sp26/fyp/experiments/osl_words_1gpu_20260417_013329/test_predictions.csv"
        label_map = None   # or "/home/sign_lang_fyp_sp26/fyp/label_map.json"
        out_dir = None
        top_k = 15
    return Args()


def load_label_map(label_map_path):
    if label_map_path is None:
        return None
    with open(label_map_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    # Normalize keys if numeric labels are stored as strings in JSON
    normalized = {}
    for k, v in data.items():
        try:
            normalized[int(k)] = v
        except Exception:
            normalized[k] = v
    return normalized


def apply_label_map(df, label_map):
    if label_map is None:
        return df

    def map_value(x):
        try:
            xi = int(x)
            return label_map.get(xi, x)
        except Exception:
            return label_map.get(x, x)

    df["true_label_name"] = df["true_label"].apply(map_value)
    df["pred_label_name"] = df["pred_label"].apply(map_value)
    return df


def parse_train_log(log_path):
    """
    Tries to extract:
    - epoch
    - lr
    - train loss
    - val loss
    - accuracy
    from free-form train.log
    """

    if not log_path.exists():
        return pd.DataFrame()

    epoch_rows = []

    # Very flexible regex patterns
    epoch_patterns = [
        re.compile(r"epoch\s*[:=\[]?\s*(\d+)", re.IGNORECASE),
        re.compile(r"Epoch\s*\[(\d+)", re.IGNORECASE),
    ]
    lr_patterns = [
        re.compile(r"\blr\b\s*[:=]\s*([0-9.eE+-]+)"),
        re.compile(r"learning[_ ]rate\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
    ]
    train_loss_patterns = [
        re.compile(r"train[_ ]?loss\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
        re.compile(r"loss\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
    ]
    val_loss_patterns = [
        re.compile(r"val[_ ]?loss\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
        re.compile(r"valid[_ ]?loss\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
    ]
    acc_patterns = [
        re.compile(r"\bacc(?:uracy)?\b\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
        re.compile(r"top1\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
        re.compile(r"val[_ ]?acc(?:uracy)?\s*[:=]\s*([0-9.eE+-]+)", re.IGNORECASE),
    ]

    current = defaultdict(lambda: None)

    def extract_first(patterns, line):
        for pat in patterns:
            m = pat.search(line)
            if m:
                try:
                    return float(m.group(1))
                except Exception:
                    return m.group(1)
        return None

    with open(log_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            epoch_num = None
            for pat in epoch_patterns:
                m = pat.search(line)
                if m:
                    try:
                        epoch_num = int(m.group(1))
                    except Exception:
                        pass
                    break

            lr = extract_first(lr_patterns, line)
            train_loss = extract_first(train_loss_patterns, line)
            val_loss = extract_first(val_loss_patterns, line)
            acc = extract_first(acc_patterns, line)

            # If this line starts a new epoch, save previous row if meaningful
            if epoch_num is not None:
                if current.get("epoch") is not None:
                    epoch_rows.append(dict(current))
                    current = defaultdict(lambda: None)
                current["epoch"] = epoch_num

            if lr is not None:
                current["lr"] = lr
            if train_loss is not None:
                current["train_loss"] = train_loss
            if val_loss is not None:
                current["val_loss"] = val_loss
            if acc is not None:
                current["accuracy"] = acc

    if current.get("epoch") is not None:
        epoch_rows.append(dict(current))

    if not epoch_rows:
        return pd.DataFrame()

    df = pd.DataFrame(epoch_rows).drop_duplicates(subset=["epoch"], keep="last")
    df = df.sort_values("epoch").reset_index(drop=True)
    return df


def compute_per_word_stats(df):
    rows = []
    grouped = df.groupby("true_label_name")

    for label, g in grouped:
        total = len(g)
        correct = int((g["true_label_name"] == g["pred_label_name"]).sum())
        wrong = total - correct
        acc = correct / total if total > 0 else 0.0

        wrong_preds = (
            g[g["true_label_name"] != g["pred_label_name"]]["pred_label_name"]
            .value_counts()
            .head(5)
            .to_dict()
        )

        rows.append({
            "word": label,
            "total_samples": total,
            "correct": correct,
            "wrong": wrong,
            "accuracy": acc,
            "top_confusions": wrong_preds,
        })

    stats_df = pd.DataFrame(rows).sort_values(
        by=["accuracy", "total_samples"], ascending=[False, False]
    ).reset_index(drop=True)

    return stats_df


def save_confusion_matrix(df, out_dir):
    labels = sorted(df["true_label_name"].astype(str).unique().tolist())
    cm = confusion_matrix(df["true_label_name"], df["pred_label_name"], labels=labels)

    fig_size = max(12, min(30, int(len(labels) * 0.35)))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, xticks_rotation=90, colorbar=False)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    out_path = out_dir / "confusion_matrix.png"
    plt.savefig(out_path, dpi=220)
    plt.close()
    return out_path


def save_training_curves(log_df, out_dir):
    saved = []

    if log_df.empty:
        return saved

    if "train_loss" in log_df.columns and log_df["train_loss"].notna().any():
        plt.figure(figsize=(8, 5))
        plt.plot(log_df["epoch"], log_df["train_loss"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Train Loss")
        plt.title("Train Loss vs Epoch")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        p = out_dir / "train_loss_curve.png"
        plt.savefig(p, dpi=180)
        plt.close()
        saved.append(p)

    if "val_loss" in log_df.columns and log_df["val_loss"].notna().any():
        plt.figure(figsize=(8, 5))
        plt.plot(log_df["epoch"], log_df["val_loss"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Validation Loss")
        plt.title("Validation Loss vs Epoch")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        p = out_dir / "val_loss_curve.png"
        plt.savefig(p, dpi=180)
        plt.close()
        saved.append(p)

    if "accuracy" in log_df.columns and log_df["accuracy"].notna().any():
        plt.figure(figsize=(8, 5))
        plt.plot(log_df["epoch"], log_df["accuracy"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.title("Accuracy vs Epoch")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        p = out_dir / "accuracy_curve.png"
        plt.savefig(p, dpi=180)
        plt.close()
        saved.append(p)

    if "lr" in log_df.columns and log_df["lr"].notna().any():
        plt.figure(figsize=(8, 5))
        plt.plot(log_df["epoch"], log_df["lr"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("Learning Rate")
        plt.title("Learning Rate vs Epoch")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        p = out_dir / "lr_curve.png"
        plt.savefig(p, dpi=180)
        plt.close()
        saved.append(p)

    return saved


def main():
    args = parse_args()

    run_dir = Path(args.run_dir)
    pred_csv = Path(args.pred_csv)
    out_dir = Path(args.out_dir) if args.out_dir else (run_dir / "analysis")
    out_dir.mkdir(parents=True, exist_ok=True)

    log_path = run_dir / "train.log"
    label_map = load_label_map(args.label_map)

    # -----------------------------
    # Load predictions
    # -----------------------------
    df = pd.read_csv(pred_csv)

    needed_cols = {"true_label", "pred_label"}
    if not needed_cols.issubset(df.columns):
        raise ValueError(f"Prediction CSV must contain columns: {needed_cols}")

    if "video_id" not in df.columns:
        df["video_id"] = np.arange(len(df))

    df = apply_label_map(df, label_map)

    if "true_label_name" not in df.columns:
        df["true_label_name"] = df["true_label"].astype(str)
    if "pred_label_name" not in df.columns:
        df["pred_label_name"] = df["pred_label"].astype(str)

    # -----------------------------
    # Overall metrics
    # -----------------------------
    overall_acc = accuracy_score(df["true_label_name"], df["pred_label_name"])
    report = classification_report(
        df["true_label_name"],
        df["pred_label_name"],
        output_dict=True,
        zero_division=0
    )
    report_df = pd.DataFrame(report).transpose()

    # -----------------------------
    # Per-word metrics
    # -----------------------------
    per_word_df = compute_per_word_stats(df)

    good_words = per_word_df.sort_values(
        by=["accuracy", "total_samples"], ascending=[False, False]
    ).head(args.top_k)

    bad_words = per_word_df.sort_values(
        by=["accuracy", "total_samples"], ascending=[True, False]
    ).head(args.top_k)

    # -----------------------------
    # Parse training log
    # -----------------------------
    log_df = parse_train_log(log_path)

    # Best epoch by accuracy if available
    best_epoch_info = {}
    if not log_df.empty and "accuracy" in log_df.columns and log_df["accuracy"].notna().any():
        best_row = log_df.loc[log_df["accuracy"].astype(float).idxmax()]
        best_epoch_info = {
            "best_epoch_by_accuracy": int(best_row["epoch"]),
            "best_accuracy": float(best_row["accuracy"]),
        }

    # -----------------------------
    # Save outputs
    # -----------------------------
    df.to_csv(out_dir / "predictions_with_names.csv", index=False)
    report_df.to_csv(out_dir / "classification_report.csv")
    per_word_df.to_csv(out_dir / "per_word_stats.csv", index=False)
    good_words.to_csv(out_dir / "good_words.csv", index=False)
    bad_words.to_csv(out_dir / "bad_words.csv", index=False)
    if not log_df.empty:
        log_df.to_csv(out_dir / "training_log_summary.csv", index=False)

    cm_path = save_confusion_matrix(df, out_dir)
    curve_paths = save_training_curves(log_df, out_dir)

    # -----------------------------
    # Write summary txt
    # -----------------------------
    summary_lines = []
    summary_lines.append("RUN ANALYSIS SUMMARY")
    summary_lines.append("=" * 60)
    summary_lines.append(f"Run directory: {run_dir}")
    summary_lines.append(f"Prediction CSV: {pred_csv}")
    summary_lines.append(f"Output directory: {out_dir}")
    summary_lines.append("")
    summary_lines.append(f"Total evaluated samples: {len(df)}")
    summary_lines.append(f"Overall accuracy: {overall_acc:.4f}")
    summary_lines.append(f"Number of classes/words: {df['true_label_name'].nunique()}")
    summary_lines.append("")

    if best_epoch_info:
        summary_lines.append(f"Best epoch by accuracy: {best_epoch_info['best_epoch_by_accuracy']}")
        summary_lines.append(f"Best accuracy from log: {best_epoch_info['best_accuracy']:.4f}")
        summary_lines.append("")

    if not good_words.empty:
        summary_lines.append(f"Top {min(args.top_k, len(good_words))} GOOD words:")
        for _, row in good_words.iterrows():
            summary_lines.append(
                f"  - {row['word']}: acc={row['accuracy']:.4f}, "
                f"correct={row['correct']}/{row['total_samples']}"
            )
        summary_lines.append("")

    if not bad_words.empty:
        summary_lines.append(f"Top {min(args.top_k, len(bad_words))} BAD words:")
        for _, row in bad_words.iterrows():
            summary_lines.append(
                f"  - {row['word']}: acc={row['accuracy']:.4f}, "
                f"correct={row['correct']}/{row['total_samples']}, "
                f"confused_with={row['top_confusions']}"
            )
        summary_lines.append("")

    summary_lines.append("Saved files:")
    summary_lines.append(f"  - {out_dir / 'classification_report.csv'}")
    summary_lines.append(f"  - {out_dir / 'per_word_stats.csv'}")
    summary_lines.append(f"  - {out_dir / 'good_words.csv'}")
    summary_lines.append(f"  - {out_dir / 'bad_words.csv'}")
    summary_lines.append(f"  - {cm_path}")
    for p in curve_paths:
        summary_lines.append(f"  - {p}")

    summary_txt = "\n".join(summary_lines)
    with open(out_dir / "summary.txt", "w", encoding="utf-8") as f:
        f.write(summary_txt)

    print(summary_txt)


if __name__ == "__main__":
    main()

AttributeError: 'Args' object has no attribute 'run_dir'

In [34]:
import os
os.system(f"nvidia-smi > {EXP_DIR}/gpu_before.txt")

0

In [1]:
os.system(f"nvidia-smi > {exp_dir}/gpu_after.txt")

NameError: name 'os' is not defined

In [1]:
from pathlib import Path
import os
import sys
import subprocess

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != "OSL_Run_Pipeline":
    alt = NOTEBOOK_DIR / "OSL_Run_Pipeline"
    if alt.exists():
        NOTEBOOK_DIR = alt

os.chdir(NOTEBOOK_DIR)
print(f"Notebook working directory: {NOTEBOOK_DIR}")
print(f"Python executable: {sys.executable}")

Notebook working directory: /home/sign_lang_fyp_sp26/fyp/Uni-sign-main/Uni-Sign-main/OSL_Run_Pipeline
Python executable: /home/sign_lang_fyp_sp26/torchenv/bin/python


In [2]:
import re

def run_step(title: str, script_name: str):
    print('=' * 80)
    print(title)
    print('=' * 80)

    cmd = [sys.executable, script_name]
    print('Command:', ' '.join(cmd), flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=NOTEBOOK_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
    )

    # Patterns for important lines we always want to show
    important_patterns = [
        r'accuracy',
        r'Epoch:.*Total time',
        r'best_checkpoint',
        r'max_accuracy',
        r'bleu|BLEU',
        r'test_loss',
        r'Evaluate',
        r'loading model',
        r'loading dataset',
        r'number of params',
        r'Start training',
        r'Training complete',
    ]
    important_re = re.compile('|'.join(important_patterns), re.IGNORECASE)

    # Pattern for noisy per-step training logs: [0/20]  [  10/5842]  eta: ...
    step_re = re.compile(r'\[\d+/\d+\]\s+\[\s*(\d+)/(\d+)\]')

    # Pattern for loading weights progress bar lines
    loading_re = re.compile(r'Loading weights|Materializing param')

    step_count = 0
    last_step_line = ""
    SHOW_EVERY = 500  # show a progress line every N steps

    if process.stdout is not None:
        for line in iter(process.stdout.readline, ''):
            if not line:
                break

            # Always show important lines (accuracy, epoch summary, etc.)
            if important_re.search(line):
                if step_count > 0:
                    print(f"  ... ({step_count} steps) {last_step_line.strip()}", flush=True)
                    step_count = 0
                print(line, end='', flush=True)
            # Suppress loading weights progress bar spam
            elif loading_re.search(line):
                continue
            # Show periodic step progress instead of every single step
            elif step_re.search(line):
                step_count += 1
                last_step_line = line
                m = step_re.search(line)
                current_step = int(m.group(1))
                if current_step % SHOW_EVERY == 0 or current_step == int(m.group(2)):
                    print(f"  [step {current_step}/{m.group(2)}] {line.strip()}", flush=True)
                    step_count = 0
            # Show everything else (warnings, errors, config, etc.)
            else:
                if step_count > 0:
                    print(f"  ... ({step_count} steps) {last_step_line.strip()}", flush=True)
                    step_count = 0
                print(line, end='')

    # Flush any remaining suppressed steps
    if step_count > 0:
        print(f"  ... ({step_count} steps) {last_step_line.strip()}", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Step failed: {script_name} (exit={return_code})")

    print(f"\nStep completed: {script_name}")

## Step 1: Install Dependencies (One Time)

In [3]:
# Uncomment to run once
run_step('Installing dependencies', 'install_dependencies.py')

Installing dependencies
Command: /home/sign_lang_fyp_sp26/torchenv/bin/python install_dependencies.py

SIGN LANGUAGE RECOGNITION - DEPENDENCY INSTALLER

✓ Python version: 3.12.3 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:46:43) [GCC 11.2.0]

Checking installed packages...
  PyTorch         ✓ installed
  OpenCV          ✓ installed
  MediaPipe       ✗ NOT installed
  NumPy           ✓ installed

Install missing packages? (y/n): Traceback (most recent call last):
  File "/home/sign_lang_fyp_sp26/fyp/Uni-sign-main/Uni-Sign-main/OSL_Run_Pipeline/install_dependencies.py", line 151, in <module>
    main()
  File "/home/sign_lang_fyp_sp26/fyp/Uni-sign-main/Uni-Sign-main/OSL_Run_Pipeline/install_dependencies.py", line 78, in main
    response = input("Install missing packages? (y/n): ").strip().lower()
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
EOFError: EOF when reading a line


RuntimeError: Step failed: install_dependencies.py (exit=1)

## Preflight: Verify MT5 Model Availability
Run the next cell once before training to ensure `google/mt5-base` is available in cache.

In [4]:
from transformers import MT5ForConditionalGeneration, T5Tokenizer

model_name = 'google/mt5-base'
print(f'Checking model: {model_name}')

try:
    _ = MT5ForConditionalGeneration.from_pretrained(model_name, use_safetensors=True)
    _ = T5Tokenizer.from_pretrained(model_name)
    print('MT5 is available (safetensors path).')
except Exception as exc:
    print(f'Safetensors path failed: {exc}')
    print('Trying default checkpoint format...')
    _ = MT5ForConditionalGeneration.from_pretrained(model_name)
    _ = T5Tokenizer.from_pretrained(model_name)
    print('MT5 is available (default checkpoint path).')

/home/sign_lang_fyp_sp26/torchenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checking model: google/mt5-base


/home/sign_lang_fyp_sp26/torchenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Safetensors path failed: Can't load the model for 'google/mt5-base'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'google/mt5-base' is the correct path to a directory containing a file named pytorch_model.bin, tf_model.h5, model.ckpt or flax_model.msgpack.
Trying default checkpoint format...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


MT5 is available (default checkpoint path).


In [5]:
import torch

def parse_version(version_str: str):
    base = version_str.split('+')[0]
    parts = base.split('.')
    major = int(parts[0]) if len(parts) > 0 else 0
    minor = int(parts[1]) if len(parts) > 1 else 0
    patch = int(parts[2]) if len(parts) > 2 else 0
    return (major, minor, patch)

current = parse_version(torch.__version__)
required = (2, 6, 0)

print('Detected torch version:', torch.__version__)

if current < required:
    print('\nTorch is below 2.6.0, which can fail when transformers loads .bin checkpoints.')
    print('Run this in a terminal, then restart the notebook kernel:')
    print('  pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121')
    raise RuntimeError('Please upgrade torch to >=2.6.0 and rerun this notebook.')

print('Torch version is compatible for training.')

Detected torch version: 2.7.1+cu118
Torch version is compatible for training.


## Step 2: Train OSL-Words (ISLR)

In [7]:
run_step('Training OSL-Words (ISLR)', 'run_osl_training.py')

Training OSL-Words (ISLR)
Command: /home/sign_lang_fyp_sp26/torchenv/bin/python run_osl_training.py
/home/sign_lang_fyp_sp26/torchenv/lib/python3.12/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
Not using distributed mode
Namespace(batch_size=4, gradient_accumulation_steps=2, gradient_clipping=1.0, epochs=20, start_epoch=0, w

RuntimeError: Step failed: run_osl_training.py (exit=1)

## Step 3: Prepare OSL-Sentences Dataset

In [5]:
run_step('Preparing OSL-Sentences dataset', 'prepare_osl_sentences_dataset.py')

Preparing OSL-Sentences dataset
Command: c:\Users\MOBPC\anaconda3\envs\torchgpu\python.exe prepare_osl_sentences_dataset.py
Prepared OSL-Sentences for Uni-Sign
  train: groups=297, entries=7536, linked=7536
  dev: groups=64, entries=70, linked=70
  test: groups=63, entries=63, linked=63
  augmented_in_train=True
  label_dir=C:\Users\MOBPC\Downloads\FYP\FYPproject\Uni-Sign-main\Uni-Sign-main\data\OSL-Sentences

Step completed: prepare_osl_sentences_dataset.py


## Step 4: Train OSL-Sentences (SLT)

In [6]:
run_step('Training OSL-Sentences (SLT)', 'run_osl_sentences_training.py')

Training OSL-Sentences (SLT)
Command: c:\Users\MOBPC\anaconda3\envs\torchgpu\python.exe run_osl_sentences_training.py
Resuming Sentences from checkpoint_14.pth (start_epoch=15, total_epochs=20)
Not using distributed mode
Namespace(batch_size=2, gradient_accumulation_steps=4, gradient_clipping=1.0, epochs=20, start_epoch=15, world_size=1, dist_url='env://', local_rank=0, hidden_dim=256, finetune='C:\\Users\\MOBPC\\Downloads\\FYP\\FYPproject\\Uni-Sign-main\\Uni-Sign-main\\out\\osl_sentences_finetuning\\checkpoint_14.pth', opt='AdamW', opt_eps=1e-09, opt_betas=None, clip_grad=None, momentum=0.9, weight_decay=0.0001, sched='cosine', lr=0.0001, min_lr=1e-08, warmup_epochs=0.0, output_dir='C:\\Users\\MOBPC\\Downloads\\FYP\\FYPproject\\Uni-Sign-main\\Uni-Sign-main\\out\\osl_sentences_finetuning', seed=42, eval=False, num_workers=4, pin_mem=True, offload=False, dtype='bf16', zero_stage=0, compute_fp32_loss=False, quick_break=0, rgb_support=False, max_length=128, dataset='OSL-Sentences', task='

KeyboardInterrupt: 

from transformers import MT5ForConditionalGeneration, T5Tokenizer

model_name = 'google/mt5-base'
print(f'Checking model: {model_name}')
print('Using use_safetensors=False to avoid HF auto-conversion API failures on some networks.')

_ = MT5ForConditionalGeneration.from_pretrained(model_name, use_safetensors=False)
_ = T5Tokenizer.from_pretrained(model_name)
print('MT5 model/tokenizer are available.')

In [ ]:
# Uncomment to run all steps
# run_step('Installing dependencies', 'install_dependencies.py')
# run_step('Training OSL-Words (ISLR)', 'run_osl_training.py')
# run_step('Preparing OSL-Sentences dataset', 'prepare_osl_sentences_dataset.py')
# run_step('Training OSL-Sentences (SLT)', 'run_osl_sentences_training.py')